# Autoregressive stages: Thinker and Talker

## Learning goals
- Understand multiple AR decoders in one request
- See how hidden states become inter-stage payloads
- Distinguish text tokens from audio codec tokens

> **Mac track:** executable cells use lightweight simulations. Boxes labelled **Linux GPU lab** show how the same concept maps to the official runtime.

## Qwen-Omni-style speech path

```text
multimodal input -> Thinker -> text + hidden states -> Talker -> audio codes -> Code2Wav
```

Both Thinker and Talker are autoregressive, but their vocabularies and outputs differ.

In [ ]:
from omni_tutorial import build_voice_pipeline
graph=build_voice_pipeline(); result,trace=graph.run("What is in this scene?")
for e in trace:
    print(f"{e['stage']:8} ({e['kind']}) output:",e['output'])

The hidden representation is an internal dependency, not necessarily a client response. A model-specific processor selects, projects, reshapes, or annotates it before the next stage.

In [ ]:
def thinker_to_talker(thinker_output):
    hidden=thinker_output["hidden"]
    return {"conditioning":[x/3 for x in hidden],"text":thinker_output["text"]}
print(thinker_to_talker({"hidden":[1,2,3],"text":"hello"}))

## Source lab

Trace `model_executor/models/qwen3_omni/`, its deployment/stage configuration, and stage-input processors. Find where Thinker output becomes Talker input.